In [34]:
pip install google-api-python-client

Note: you may need to restart the kernel to use updated packages.


### NEW BLOK

In [3]:
pip install isodate

Note: you may need to restart the kernel to use updated packages.


In [35]:
# STEP 0: Import necessary libraries
# These are like bringing in tools you'll need for your work.
import re                  # 're' is for Regular Expressions, used to find patterns in text (like YouTube video IDs in links).
from googleapiclient.discovery import build # This is part of Google's official library to talk to Google APIs (like YouTube's).
from googleapiclient.errors import HttpError # This helps us catch specific errors if the API doesn't respond correctly.
import isodate             # 'isodate' is a special tool to understand date/time formats like those used by YouTube for video durations.
import pandas as pd

from dotenv import load_dotenv
import os

# STEP 1: Set up your YouTube Data API Key
# This is like your personal access card to YouTube's data.
# IMPORTANT: You MUST replace "YOUR_YOUTUBE_DATA_API_KEY" with your actual API key.
# Get your API key from Google Cloud Console (APIs & Services -> Credentials).
load_dotenv(override=True)
api_key = os.environ['API_KEY']


In [36]:
# STEP 2: Open file with YouTube links

df = pd.read_csv('/Users/katemamyan/Documents/GitHub/LEDE_PROGRAM/project_01/cleaned_archive_data.csv', index_col=False)
df.head()

,link,video_link,title,time,date,day,arm_month,year,month,iso_date,cleaned_title
0,https://www.1tv.am/hy/video/Հարցազրույց-Սոսի-Թ...,https://www.youtube.com/embed/gkLMht3Vt-k?auto...,Հարցազրույց Սոսի Թաթիկյանի հետ,22:10,"03 Հլս, 2025",3,Հլս,2025,7,2025-07-03,սոս թաթիկյան
1,https://www.1tv.am/hy/video/Հարցազրույց-Համբիկ...,https://www.youtube.com/embed/yjj8ISwxTwE?auto...,Հարցազրույց Համբիկ Սարաֆյանի հետ,22:10,"02 Հլս, 2025",2,Հլս,2025,7,2025-07-02,համբիկ սարաֆյան
2,https://www.1tv.am/hy/video/Հարցազրույց-Կայա-Կ...,https://www.youtube.com/embed/Hd6l_IRhygI?auto...,Հարցազրույց Կայա Կալասի հետ,22:10,"01 Հլս, 2025",1,Հլս,2025,7,2025-07-01,կայա կալաս
3,https://www.1tv.am/hy/video/Հարցազրույց-Արթուր...,https://www.youtube.com/embed/d5x7uI6cCLI?auto...,Հարցազրույց Արթուր Պողոսյանի հետ,22:10,"30 Հնս, 2025",30,Հնս,2025,6,2025-06-30,արթուր պողոսյան
4,https://www.1tv.am/hy/video/Հարցազրույց-Էլեն-Հ...,https://www.youtube.com/embed/gpIDqXFZx8o?auto...,Հարցազրույց Էլեն Հոխիկյանի հետ,22:30,"27 Հնս, 2025",27,Հնս,2025,6,2025-06-27,էլեն հոխիկյան


In [37]:

# STEP 3: Define a helper function to get the video ID from a YouTube link
# This function takes a full YouTube link and pulls out the unique video ID (e.g., 'dQw4w9WgXcQ').
def get_video_id(youtube_link):
    """
    Extracts the unique 11-character video ID from various YouTube URL formats.
    Returns the video ID string if found, otherwise returns None.
    """
    # This is a regular expression pattern. It's designed to find the 11-character
    # video ID that comes after common YouTube link parts like "v=", "youtu.be/", etc.
    match = re.search(
        r"(?:v=|youtu\.be/|embed/|shorts/|(?:\w+\.)?youtube\.com/clip/)([a-zA-Z0-9_-]{11})",
        youtube_link
    )
    # If the pattern is found in the link:
    if match:
        # We return the found ID (the part inside the parentheses in the regex).
        return match.group(1)
    # If the pattern is not found (meaning it's not a recognizable YouTube video link):
    return None


In [38]:

# STEP 4: Define a helper function to convert YouTube's duration format
# YouTube API gives duration in a special format (e.g., "PT1H2M3S" for 1 hour, 2 minutes, 3 seconds).
# This function converts that into a more readable "HH:MM:SS" format.
def parse_iso8601_duration(iso_duration):
    """
    Parses an ISO 8601 duration string (e.g., 'PT1H2M3S') into HH:MM:SS format.
    Returns 'N/A' if the string cannot be parsed.
    """
    try:
        # Use the 'isodate' library to understand the special duration string.
        duration_obj = isodate.parse_duration(iso_duration)
        # Convert the parsed duration into a total number of seconds.
        total_seconds = int(duration_obj.total_seconds())

        # Calculate hours, minutes, and remaining seconds from the total seconds.
        hours, remainder = divmod(total_seconds, 3600) # 3600 seconds in an hour
        minutes, seconds = divmod(remainder, 60)      # 60 seconds in a minute

        # Format the time as a string like "01:02:03".
        # ":02d" means "format as a decimal integer, padded with a leading zero if needed, to 2 digits".
        return f"{hours:02d}:{minutes:02d}:{seconds:02d}"
    except Exception:
        # If anything goes wrong during parsing (e.g., the string is not a valid duration),
        # return "N/A" (Not Available).
        return "N/A"


In [39]:

# STEP 5: Define the main function to fetch video data from YouTube API
# This function will ask the YouTube API for the duration and thumbnail for each video ID.
# --- CORRECTED get_video_duration_and_thumbnail function ---
def get_video_duration_and_thumbnail(video_ids_list, api_key_param):
    """
    Fetches video duration, thumbnail URLs, AND VIEWS for a list of YouTube video IDs.

    Args:
        video_ids_list (list): A list of 11-character YouTube video IDs.
        api_key_param (str): Your YouTube Data API v3 key.

    Returns:
        dict: A dictionary where keys are video IDs and values are dictionaries
              containing 'duration' (HH:MM:SS string), 'thumbnails' (dict of URLs),
              and 'views' (integer).
              Returns 'N/A' or empty dicts if data is not available or an error occurs.
    """
    youtube_service = build("youtube", "v3", developerKey=api_key_param)
    all_video_data_results = {}

    for i in range(0, len(video_ids_list), 50):
        batch_ids = video_ids_list[i:i+50]
        try:
            request = youtube_service.videos().list(
                # <<< CRITICAL FIX: Added "statistics" to the part parameter >>>
                part="statistics,contentDetails,snippet",
                id=",".join(batch_ids)
            )
            response = request.execute()

            for item in response.get("items", []):
                video_id = item["id"]
                
                # --- Extract Duration ---
                content_details = item.get("contentDetails", {})
                duration_iso = content_details.get("duration", "N/A")
                formatted_duration = parse_iso8601_duration(duration_iso)

                # --- Extract Thumbnails ---
                snippet = item.get("snippet", {})
                thumbnails = snippet.get("thumbnails", {})
                
                current_video_thumbnail_urls = {
                    quality: data.get("url")
                    for quality, data in thumbnails.items()
                }

                # --- Extract Views --- <<< NEW/FIXED PART
                statistics = item.get("statistics", {})
                # 'viewCount' is a string, convert it to an integer. Default to 0 if not found.
                views_count = int(statistics.get("viewCount", 0)) 

                all_video_data_results[video_id] = {
                    "duration": formatted_duration,
                    "thumbnails": current_video_thumbnail_urls,
                    "views": views_count # <<< Added views here
                }

        except HttpError as e:
            print(f"ERROR: HTTP error {e.resp.status} occurred for batch {batch_ids}: {e.content}")
            for vid_id in batch_ids:
                if vid_id not in all_video_data_results:
                    all_video_data_results[vid_id] = {"duration": "N/A", "thumbnails": {}, "views": "N/A"}
        except Exception as e:
            print(f"ERROR: An unexpected error occurred for batch {batch_ids}: {e}")
            for vid_id in batch_ids:
                if vid_id not in all_video_data_results:
                    all_video_data_results[vid_id] = {"duration": "N/A", "thumbnails": {}, "views": "N/A"}
    
    return all_video_data_results
# --- END OF CORRECTED FUNCTION ---
    


In [43]:

print("Program started: Adding YouTube data (including views) to DataFrame and saving to CSV.")

# --- STEP 6: Get YouTube links from your DataFrame's column ---
# We convert the 'video_link' column from your DataFrame into a Python list.
youtube_links_to_process = df['video_link'].to_list()

# --- STEP 7: Extract video IDs and add a 'video_id' column to DataFrame ---
# It's more efficient to apply the extraction directly to the DataFrame.
print("Extracting video IDs from links and adding to DataFrame...")
df['video_id'] = df['video_link'].apply(get_video_id)
print("Video IDs extracted and added to 'video_id' column.")

# --- STEP 8: Get a list of unique, valid video IDs for API calls ---
# We only want to send valid, non-duplicate IDs to the YouTube API.
# Filter out any None values (from links where an ID couldn't be extracted).
valid_video_ids = [vid for vid in df['video_id'].unique() if vid]

if not valid_video_ids:
    print("No valid YouTube video IDs found in your DataFrame. No data will be fetched from API.")
    # If no valid IDs, add empty columns to the DataFrame to prevent errors later when saving.
    df['Duration'] = 'N/A'
    df['Thumbnail_URL'] = 'N/A'
    df['Views'] = 'N/A' # <<< NEW: Initialize 'Views' column
else:
    # --- STEP 9: Fetch duration and thumbnail data from YouTube API ---
    # Our get_video_duration_and_thumbnail function already fetches 'statistics' which includes 'viewCount'.
    print(f"\nFetching duration, thumbnails, and views for {len(valid_video_ids)} unique videos...")
    # Call our function to get the data from YouTube.
    # 'all_videos_details' will be a dictionary where keys are video IDs.
    all_videos_details = get_video_duration_and_thumbnail(valid_video_ids, api_key) # Using API_KEY as defined in the global scope
    print("Data fetching complete.")

    # --- STEP 10: Add new 'Duration', 'Thumbnail_URL', AND 'Views' columns to your DataFrame ---
    print("Adding fetched data to DataFrame as new columns (Duration, Thumbnail_URL, Views)...")

    # Add 'Duration' column:
    df['Duration'] = df['video_id'].map(lambda x: all_videos_details.get(x, {}).get('duration', 'N/A'))

    # Add 'Thumbnail_URL' column:
    df['Thumbnail_URL'] = df['video_id'].map(
        lambda x: all_videos_details.get(x, {}).get('thumbnails', {}).get('high', all_videos_details.get(x, {}).get('thumbnails', {}).get('default', 'N/A'))
    )

    # <<< NEW: Add 'Views' column >>>
    # The 'views' count is directly under the 'statistics' part of the API response,
    # which is captured by our get_video_duration_and_thumbnail function.
    df['Views'] = df['video_id'].map(lambda x: all_videos_details.get(x, {}).get('views', 'N/A'))
    # You might want to convert 'Views' to a numeric type if it's not 'N/A'
    # df['Views'] = pd.to_numeric(df['Views'], errors='coerce').fillna('N/A')

    print("New 'Duration', 'Thumbnail_URL', and 'Views' columns added to DataFrame.")

    # --- OPTIONAL: Clean up the 'video_id' column ---
    # If you don't need the temporary 'video_id' column in your final CSV, you can remove it.
    # Uncomment the line below if you want to drop it:
    # df = df.drop(columns=['video_id'])

print("\n--- Displaying Updated DataFrame (first 10 rows) ---")
print(df.head(10)) # Print more rows to see the new columns effectively.
print("-" * 50)

# --- STEP 11: Save the updated DataFrame to a CSV file ---
output_filename = "youtube_video_data_with_all_details.csv" # Changed filename to reflect all data

df['iso_date'] = pd.to_datetime(df['iso_date']) # Making column date
filtered_df = df[df['iso_date'] >= pd.Timestamp('2021-06-20')] # Filter the data, keep a part from last elections in 2021, June 20

# 'index=False' prevents pandas from writing the DataFrame's row numbers into the CSV file.
df.to_csv(output_filename, index=False)
filtered_df.to_csv("data_to_analyze_2021-2025.csv" , index=False)

print(f"\nUpdated DataFrame saved to '{output_filename}' successfully!")

print("\n--- Program Finished ---")

Program started: Adding YouTube data (including views) to DataFrame and saving to CSV.
Extracting video IDs from links and adding to DataFrame...
Video IDs extracted and added to 'video_id' column.

Fetching duration, thumbnails, and views for 1262 unique videos...
Data fetching complete.
Adding fetched data to DataFrame as new columns (Duration, Thumbnail_URL, Views)...
New 'Duration', 'Thumbnail_URL', and 'Views' columns added to DataFrame.

--- Displaying Updated DataFrame (first 10 rows) ---
                                                link  \
0  https://www.1tv.am/hy/video/Հարցազրույց-Սոսի-Թ...   
1  https://www.1tv.am/hy/video/Հարցազրույց-Համբիկ...   
2  https://www.1tv.am/hy/video/Հարցազրույց-Կայա-Կ...   
3  https://www.1tv.am/hy/video/Հարցազրույց-Արթուր...   
4  https://www.1tv.am/hy/video/Հարցազրույց-Էլեն-Հ...   
5  https://www.1tv.am/hy/video/Հարցազրույց-Խորեն-...   
6  https://www.1tv.am/hy/video/Հարցազրույց-Անդրան...   
7  https://www.1tv.am/hy/video/Հարցազրույց-Ալեն-Ս..